In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta

In [2]:
ticker = "VOO"
data = yf.download(ticker, period="1mo", interval="30m")

C:\venv\Investment\investment-311v1\Lib\site-packages\yfinance\scrapers\history.py:239: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  quotes2 = quotes.resample('30T')
[*********************100%%**********************]  1 of 1 completed


In [3]:
data["VWAP"] = ta.volume.volume_weighted_average_price(
    data.High, data.Low, data["Adj Close"], data.Volume
)
data["RSI"] = ta.momentum.RSIIndicator(data["Adj Close"], window=16, fillna=True).rsi()

In [4]:
def get_bollinger_bands(data, window=14, window_dev=2.0):
    bands = ta.volatility.BollingerBands(data["Adj Close"], window=window, window_dev=window_dev)
    data["BBL_" + str(window) + "_" + str(window_dev)] = bands.bollinger_lband()
    data["BBM_" + str(window) + "_" + str(window_dev)] = bands.bollinger_mavg()
    data["BBU_" + str(window) + "_" + str(window_dev)] = bands.bollinger_hband()
    data["BBP_" + str(window) + "_" + str(window_dev)] = bands.bollinger_pband()
    return data

In [5]:
data = get_bollinger_bands(data)

In [6]:
VWAPsignal = [0] * len(data)
backcandles = 15

for row in range(backcandles, len(data)):
    upt = 1
    dnt = 1
    for i in range(row - backcandles, row + 1):
        if max(data.Open[i], data.Close[i]) >= data.VWAP[i]:
            dnt = 0
        if min(data.Open[i], data.Close[i]) <= data.VWAP[i]:
            upt = 0
    if upt == 1 and dnt == 1:
        VWAPsignal[row] = 3
    elif upt == 1:
        VWAPsignal[row] = 2
    elif dnt == 1:
        VWAPsignal[row] = 1

data["VWAPSignal"] = VWAPsignal

C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\120580492.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if max(data.Open[i], data.Close[i])>=data.VWAP[i]:
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\120580492.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if min(data.Open[i], data.Close[i])<=data.VWAP[i]:


In [7]:
def TotalSignal(l):
    if data.VWAPSignal[l] == 2 and data.Close[l] <= data["BBL_14_2.0"][l] and data.RSI[l] < 45:
        return 2
    if data.VWAPSignal[l] == 1 and data.Close[l] >= data["BBU_14_2.0"][l] and data.RSI[l] > 55:
        return 1
    return 0


TotSignal = [0] * len(data)
for row in range(backcandles, len(data)):  # careful backcandles used previous cell
    TotSignal[row] = TotalSignal(row)
data["TotalSignal"] = TotSignal

C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\931924749.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (data.VWAPSignal[l]==2
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\931924749.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (data.VWAPSignal[l]==1
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\931924749.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  and data.Close[l]<=data['BBL_14_2.0'][l]
C

In [9]:
data[data["TotalSignal"] == 1]

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,BBU_14_2.0,BBP_14_2.0,VWAPSignal,TotalSignal
Datetime,,,,,,,,,,,,,,


In [8]:
ticker = "IJR"
data = yf.download(ticker, period="1y", interval="1h")


class CommonLogic:
    def __init__(self):
        pass

    def get_bollinger_bands(self, data, window=14, window_dev=2.0):
        bands = ta.volatility.BollingerBands(
            data[self.close_column], window=window, window_dev=window_dev
        )
        data["BBL_" + str(window) + "_" + str(window_dev)] = bands.bollinger_lband()
        data["BBM_" + str(window) + "_" + str(window_dev)] = bands.bollinger_mavg()
        data["BBU_" + str(window) + "_" + str(window_dev)] = bands.bollinger_hband()
        data["BBP_" + str(window) + "_" + str(window_dev)] = bands.bollinger_pband()
        return data

    def get_vwap(
        self,
        data,
    ):
        return ta.volume.volume_weighted_average_price(
            data.High, data.Low, data[self.close_column], data.Volume
        )

    def get_rsi(self, data):
        return ta.momentum.RSIIndicator(data[self.close_column], window=16, fillna=True).rsi()


class ScalpingVWAPRSI(CommonLogic):
    def __init__(self, data, close_column, debug=False):
        super().__init__()
        self.data = data
        self.close_column = close_column
        self.debug = debug

    def get_indicators(self):
        self.data["VWAP"] = self.get_vwap(self.data)
        self.data["RSI"] = self.get_rsi(self.data)
        self.data = self.get_bollinger_bands(self.data)
        return self.data

    def _get_vwap_signal(self, backcandles=15):
        VWAPsignal = [0] * len(data)
        for row in range(backcandles, len(self.data)):
            upt = 1
            dnt = 1
            for i in range(row - backcandles, row + 1):
                if max(self.data.Open[i], self.data[self.close_column][i]) >= data.VWAP[i]:
                    dnt = 0
                if min(self.data.Open[i], self.data[self.close_column][i]) <= data.VWAP[i]:
                    upt = 0
            if upt == 1 and dnt == 1:
                VWAPsignal[row] = 3
            elif upt == 1:
                VWAPsignal[row] = 2
                if self.debug:
                    print(f"Uptrend signal found at {self.data.index[row]}")
            elif dnt == 1:
                VWAPsignal[row] = 1
                if self.debug:
                    print(f"Downtrend signal found at {self.data.index[row]}")

        self.data["VWAPSignal"] = VWAPsignal

    def _get_total_signals(self, row, buy_signal=45, sell_signal=55):
        if (
            self.data.VWAPSignal[row] == 2
            and self.data[self.close_column][row] <= self.data["BBL_14_2.0"][row]
            and self.data.RSI[row] < buy_signal
        ):
            if self.debug:
                print(f"Buy signal foud at {self.data.index[row]}")
            return 2
        if (
            self.data.VWAPSignal[row] == 1
            and self.data[self.close_column][row] >= self.data["BBU_14_2.0"][row]
            and self.data.RSI[row] > sell_signal
        ):
            if self.debug:
                print(f"Sell signal foud at {self.data.index[row]}")
            return 1
        return 0

    def get_signals(self, rsi_buy_signal=45, rsi_sell_signal=55):
        self._get_vwap_signal()
        TotSignal = [0] * len(self.data)
        for row in range(backcandles, len(self.data)):  # careful backcandles used previous cell
            TotSignal[row] = self._get_total_signals(row, rsi_buy_signal, rsi_sell_signal)

        self.data["Signal"] = TotSignal
        return self.data


strategy = ScalpingVWAPRSI(data, "Adj Close", True)
data = strategy.get_indicators()
data = strategy.get_signals(rsi_buy_signal=80, rsi_sell_signal=30)
data.tail()

[*********************100%%**********************]  1 of 1 completed
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\809500081.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if max(self.data.Open[i], self.data[self.close_column][i])>=data.VWAP[i]:
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\809500081.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if min(self.data.Open[i], self.data[self.close_column][i])<=data.VWAP[i]:


Downtrend signal found at 2023-08-03 10:30:00-04:00
Downtrend signal found at 2023-08-03 11:30:00-04:00
Downtrend signal found at 2023-08-14 13:30:00-04:00
Downtrend signal found at 2023-08-14 14:30:00-04:00
Downtrend signal found at 2023-08-14 15:30:00-04:00
Downtrend signal found at 2023-08-15 09:30:00-04:00
Downtrend signal found at 2023-08-15 10:30:00-04:00
Downtrend signal found at 2023-08-15 11:30:00-04:00
Downtrend signal found at 2023-08-15 12:30:00-04:00
Downtrend signal found at 2023-08-15 13:30:00-04:00
Downtrend signal found at 2023-08-15 14:30:00-04:00
Downtrend signal found at 2023-08-15 15:30:00-04:00
Downtrend signal found at 2023-08-16 09:30:00-04:00
Downtrend signal found at 2023-08-16 10:30:00-04:00
Downtrend signal found at 2023-08-16 11:30:00-04:00
Downtrend signal found at 2023-08-16 12:30:00-04:00
Downtrend signal found at 2023-08-16 13:30:00-04:00
Downtrend signal found at 2023-08-16 14:30:00-04:00
Downtrend signal found at 2023-08-16 15:30:00-04:00
Downtrend si

C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\809500081.py:60: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (self.data.VWAPSignal[row]==2
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\809500081.py:66: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (self.data.VWAPSignal[row]==1
C:\Users\I838159\AppData\Local\Temp\ipykernel_30488\809500081.py:67: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  and self.data[self.close_

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,BBU_14_2.0,BBP_14_2.0,VWAPSignal,Signal
Datetime,,,,,,,,,,,,,,
2024-07-31 11:30:00-04:00,118.269997,119.034103,118.260002,118.964401,118.964401,378910,117.440016,69.196520,116.205908,117.437465,118.669022,1.119921,0,0
2024-07-31 12:30:00-04:00,118.949997,119.139999,118.660004,118.820000,118.820000,443291,117.551162,67.196846,116.254714,117.592465,118.930216,0.958805,0,0
2024-07-31 13:30:00-04:00,118.809998,118.809998,118.230003,118.525002,118.525002,472099,117.661305,63.215977,116.404252,117.723179,119.042107,0.803968,0,0
2024-07-31 14:30:00-04:00,118.481003,120.739998,118.481003,119.065102,119.065102,1511700,118.016745,67.030380,116.479850,117.878808,119.277766,0.923992,0,0
2024-07-31 15:30:00-04:00,118.980003,119.180000,118.239998,118.339996,118.339996,698031,118.104355,58.363474,116.659128,117.977379,119.295630,0.637537,0,0


In [9]:
data[data["Signal"] == 2]

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,BBU_14_2.0,BBP_14_2.0,VWAPSignal,Signal
Datetime,,,,,,,,,,,,,,


In [74]:
ticker = "IJR"
data = yf.download(ticker, period="1y", interval="1h")


class CommonLogic:
    def __init__(self):
        pass

    def get_bollinger_bands(self, data, window=14, window_dev=2.0):
        bands = ta.volatility.BollingerBands(
            data[self.close_column], window=window, window_dev=window_dev
        )
        data["BBL_" + str(window) + "_" + str(window_dev)] = bands.bollinger_lband()
        data["BBM_" + str(window) + "_" + str(window_dev)] = bands.bollinger_mavg()
        data["BBU_" + str(window) + "_" + str(window_dev)] = bands.bollinger_hband()
        data["BBP_" + str(window) + "_" + str(window_dev)] = bands.bollinger_pband()
        return data

    def get_vwap(self, data):
        return ta.volume.volume_weighted_average_price(
            data.High, data.Low, data[self.close_column], data.Volume
        )

    def get_rsi(self, data):
        return ta.momentum.RSIIndicator(data[self.close_column], window=16, fillna=True).rsi()


class ScalpingVWAPRSI(CommonLogic):
    def __init__(self, data, close_column, debug=False, backcandles=15):
        super().__init__()
        self.data = data
        self.close_column = close_column
        self.debug = debug
        self.backcandles = backcandles

    def get_indicators(self):
        self.data["VWAP"] = self.get_vwap(self.data)
        self.data["RSI"] = self.get_rsi(self.data)
        self.data = self.get_bollinger_bands(self.data)
        return self.data

    def _get_vwap_signal(self):
        VWAPsignal = [0] * len(self.data)
        for row in range(self.backcandles, len(self.data)):
            upt = 1
            dnt = 1
            for i in range(row - self.backcandles, row + 1):
                if max(self.data.Open[i], self.data[self.close_column][i]) >= self.data.VWAP[i]:
                    dnt = 0
                if min(self.data.Open[i], self.data[self.close_column][i]) <= self.data.VWAP[i]:
                    upt = 0
            if upt == 1 and dnt == 1:
                VWAPsignal[row] = 3
            elif upt == 1:
                VWAPsignal[row] = 2
                if self.debug:
                    print(f"Uptrend signal found at {self.data.index[row]}")
            elif dnt == 1:
                VWAPsignal[row] = 1
                if self.debug:
                    print(f"Downtrend signal found at {self.data.index[row]}")

        self.data["VWAPSignal"] = VWAPsignal

    def _get_total_signals(self, row, buy_signal=45, sell_signal=55):
        if (
            self.data.VWAPSignal[row] == 2
            # and self.data[self.close_column][row] <= self.data['BBL_14_2.0'][row]
            and self.data.RSI[row] < buy_signal
        ):
            if self.debug:
                print(f"Buy signal found at {self.data.index[row]}")
            return 2
        if (
            self.data.VWAPSignal[row] == 1
            # and self.data[self.close_column][row] >= self.data['BBU_14_2.0'][row]
            and self.data.RSI[row] > sell_signal
        ):
            if self.debug:
                print(f"Sell signal found at {self.data.index[row]}")
            return 1
        return 0

    def get_signals(self, rsi_buy_signal=45, rsi_sell_signal=55):
        self._get_vwap_signal()
        TotSignal = [0] * len(self.data)
        for row in range(backcandles, len(self.data)):
            TotSignal[row] = self._get_total_signals(row, rsi_buy_signal, rsi_sell_signal)

        self.data["Signal"] = TotSignal
        return self.data

[*********************100%%**********************]  1 of 1 completed


In [75]:
strategy = ScalpingVWAPRSI(data, "Adj Close", False)
data = strategy.get_indicators()
data = strategy.get_signals(rsi_buy_signal=80, rsi_sell_signal=30)
data.tail()

C:\Users\I838159\AppData\Local\Temp\ipykernel_36028\4167372348.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if max(self.data.Open[i], self.data[self.close_column][i]) >= self.data.VWAP[i]:
C:\Users\I838159\AppData\Local\Temp\ipykernel_36028\4167372348.py:46: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if min(self.data.Open[i], self.data[self.close_column][i]) <= self.data.VWAP[i]:
C:\Users\I838159\AppData\Local\Temp\ipykernel_36028\4167372348.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with 

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,BBU_14_2.0,BBP_14_2.0,VWAPSignal,Signal
Datetime,,,,,,,,,,,,,,
2024-07-29 11:30:00-04:00,116.959999,117.355003,116.680000,117.059998,117.059998,463884,117.059871,55.429933,115.895762,117.021822,118.147881,0.516951,0,0
2024-07-29 12:30:00-04:00,117.070000,117.160004,116.650002,116.650002,116.650002,282752,117.063812,51.976717,115.871574,117.009679,118.147784,0.341984,0,0
2024-07-29 13:30:00-04:00,116.650002,116.800003,116.480003,116.695000,116.695000,396025,117.077494,52.324429,115.937848,117.033607,118.129367,0.345492,0,0
2024-07-29 14:30:00-04:00,116.714996,116.959999,116.504997,116.886299,116.886299,393466,117.125363,53.840015,116.019314,117.066557,118.113800,0.413937,0,0
2024-07-29 15:30:00-04:00,116.889999,117.260002,116.855797,116.989998,116.989998,377156,117.232253,54.673175,116.477394,117.163700,117.850006,0.373451,0,0


In [76]:
data.to_csv("file.csv")